# RAGTruth dataset lineage audit

This notebook is a read-only audit. It reconstructs the lineage of the broad textual RAGTruth dataset, the English QA confirmatory population, and the Portuguese NLLB population. It does not create datasets, translations, splits, checkpoints, or training outputs.

The analysis uses the existing preprocessing helpers where possible. The translation-filter section re-evaluates the existing predicate in memory because the stored manifest records aggregate counts but not the trigger decomposition.

In [ ]:
from __future__ import annotations

import collections
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from ragtruth_transfer.ragtruth_data import (
    label_independent_sentence_claims,
    response_claims,
)
from ragtruth_transfer.translation_qa import classify_translation_pair
from scripts.filter_ragtruth_translation_failures import validate_filtered_dataset

BROAD = ROOT / 'data/processed/ragtruth_textual'
PT = ROOT / 'data/processed/ragtruth_textual_pt_nllb'
PT_FILTERED = ROOT / 'data/processed/ragtruth_textual_pt_nllb_filtered'
TOP4 = ROOT / 'results/ragtruth_qa_top4_label_independent/ragtruth_qa_top4_label_independent_embeddings/9873bae9953c3e29/dataset.parquet'
VIEW = ROOT / 'results/ragtruth_qa_training_view/deduplicate_source_claim_drop_conflicts_v1/0cdf598fa866741d/dataset.parquet'

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl_dir(directory: Path):
    rows = []
    for split in ('train', 'validation', 'test'):
        for line in (directory / f'{split}.jsonl').open(encoding='utf-8'):
            row = json.loads(line)
            row['_file_split'] = split
            rows.append(row)
    return rows

def sha256(path: Path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def count_field(rows, field):
    return dict(collections.Counter(str(row.get(field)) for row in rows))

broad_rows = read_jsonl_dir(BROAD)
pt_rows = read_jsonl_dir(PT)
filtered_rows = read_jsonl_dir(PT_FILTERED)
top4 = pd.read_parquet(TOP4)
view = pd.read_parquet(VIEW)
raw_responses = [json.loads(line) for line in (ROOT / 'data/raw/ragtruth/response.jsonl').open(encoding='utf-8')]
raw_sources = [json.loads(line) for line in (ROOT / 'data/raw/ragtruth/source_info.jsonl').open(encoding='utf-8')]
source_by_id = {str(row['source_id']): row for row in raw_sources}

print('READ-ONLY ROOT:', ROOT)
print('Loaded rows:', {'broad': len(broad_rows), 'pt': len(pt_rows), 'pt_filtered': len(filtered_rows), 'top4': len(top4), 'training_view': len(view)})

## 1. Population inventory and source lineage

In [ ]:
def population_row(name, rows, source_field='source_id', response_field='response_id'):
    labels = collections.Counter(int(bool(row['label'])) for row in rows if 'label' in row)
    return {
        'population': name,
        'rows': len(rows),
        'sources': len({str(row[source_field]) for row in rows if source_field in row}),
        'responses': len({str(row[response_field]) for row in rows if response_field in row}),
        'positives': labels.get(1, 0),
        'negatives': labels.get(0, 0),
        'splits': count_field(rows, '_file_split'),
    }

print(pd.DataFrame([
    population_row('broad textual', broad_rows),
    population_row('translated full PT', pt_rows),
    population_row('translated filtered PT', filtered_rows),
    {
        'population': 'English Top-4 parent',
        'rows': len(top4),
        'sources': top4.source_id.nunique(),
        'responses': top4.response_id.nunique(),
        'positives': int(top4.label.sum()),
        'negatives': int((~top4.label.astype(bool)).sum()),
        'splits': top4.split.value_counts().to_dict(),
    },
    {
        'population': 'English training view',
        'rows': len(view),
        'sources': view.source_id.nunique(),
        'responses': view.response_id.nunique(),
        'positives': int(view.label.sum()),
        'negatives': int((~view.label.astype(bool)).sum()),
        'splits': view.split.value_counts().to_dict(),
    },
]).to_string(index=False))

print('Raw source task counts:', count_field(raw_sources, 'task_type'))
print('Raw response quality counts:', count_field(raw_responses, 'quality'))
print('Raw source counts by task:', collections.Counter(str(row['task_type']) for row in raw_sources))
print('Broad task counts:', count_field(broad_rows, 'task_type'))
print('Broad task and split counts:', dict(collections.Counter((row['task_type'], row['_file_split']) for row in broad_rows)))

broad_ids = {str(row['example_id']) for row in broad_rows}
pt_ids = {str(row['example_id']) for row in pt_rows}
filtered_ids = {str(row['example_id']) for row in filtered_rows}
print('Example-ID equality broad/PT:', broad_ids == pt_ids)
print('Full PT IDs removed by filter:', len(pt_ids - filtered_ids))
print('Example-ID overlap English Top-4 vs broad:', len(set(top4.example_id.astype(str)) & broad_ids))
print('ID formats:', {'legacy': 'response_id:claim_index', 'English Top-4': 'response_id:claim_start:claim_end'})

## 2. Reconstructing the English populations from raw RAGTruth

In [ ]:
def raw_claim_rows(boundary):
    rows = []
    for response in raw_responses:
        source = source_by_id[str(response['source_id'])]
        if source.get('task_type') != 'QA' or response.get('quality', 'good') != 'good':
            continue
        if response.get('split') not in {'train', 'test'}:
            continue
        claims = (
            response_claims(response, 'claim', 3)
            if boundary == 'legacy'
            else label_independent_sentence_claims(response, 3)
        )
        for claim in claims:
            rows.append({
                'source_id': str(response['source_id']),
                'response_id': str(response['id']),
                'split': str(response['split']),
                'claim': str(claim['claim']),
                'label': bool(claim['label']),
                'claim_start': int(claim['claim_start']),
                'claim_end': int(claim['claim_end']),
                'example_id': f"{response['id']}:{claim['claim_index']}" if boundary == 'legacy' else f"{response['id']}:{claim['claim_start']}:{claim['claim_end']}",
            })
    return rows

legacy_qa = raw_claim_rows('legacy')
independent_qa = raw_claim_rows('independent')
print('Legacy QA claims:', len(legacy_qa), collections.Counter(row['split'] for row in legacy_qa))
print('Label-independent QA claims:', len(independent_qa), collections.Counter(row['split'] for row in independent_qa))
print('Label-independent reconstruction equals Top-4 metadata:', {
    'rows': len(independent_qa) == len(top4),
    'sources': len({row['source_id'] for row in independent_qa}) == top4.source_id.nunique(),
    'responses': len({row['response_id'] for row in independent_qa}) == top4.response_id.nunique(),
})

def lineage_key(row):
    return (str(row['source_id']), str(row['response_id']), str(row['claim']), int(row['claim_start']), int(row['claim_end']), bool(row['label']))

legacy_by_key = {lineage_key(row): row for row in broad_rows if row.get('task_type') == 'QA'}
parent_records = top4.to_dict('records')
exact_matches = [row for row in parent_records if lineage_key(row) in legacy_by_key]
print('English parent rows exactly matching a legacy QA tuple:', len(exact_matches), 'of', len(parent_records))
print('Rows differing because of segmentation/claim boundaries:', len(parent_records) - len(exact_matches))
print('Exact legacy boundary label pairs:', collections.Counter((bool(row['label']), bool(legacy_by_key[lineage_key(row)]['label'])) for row in exact_matches))

## 3. Training-view reduction, duplicates, and conflicts

In [ ]:
parent_group_sizes = top4.assign(_group=top4.source_id.astype(str) + '\x1f' + top4.claim.astype(str)).groupby('_group').size()
duplicate_groups = parent_group_sizes[parent_group_sizes > 1]
duplicate_audit = read_json(ROOT / 'results/ragtruth_qa_training_view/deduplicate_source_claim_drop_conflicts_v1/0cdf598fa866741d/duplicate_audit.json')
training_audit = read_json(ROOT / 'results/ragtruth_qa_training_view/deduplicate_source_claim_drop_conflicts_v1/0cdf598fa866741d/training_view_audit.json')
print('Parent duplicate groups:', len(duplicate_groups))
print('Parent duplicate rows:', int(duplicate_groups.sum()))
print('Duplicate extras:', int(duplicate_groups.sum() - len(duplicate_groups)))
print('Duplicate decisions:', duplicate_audit['decision_counts'])
print('Conflict diagnostics:', read_json(ROOT / 'results/ragtruth_qa_training_view/deduplicate_source_claim_drop_conflicts_v1/0cdf598fa866741d/manifest.json')['conflict_review']['diagnostics'])
print('Input-mismatch groups:', training_audit['input_mismatch_groups_included'])
print('Training-view rows:', len(top4), '->', len(view), 'removed:', len(top4) - len(view))
print('Removed copies:', training_audit['removed']['collapsed_copies']['rows'], 'removed conflict rows:', training_audit['removed']['conflicting_groups']['rows'])
print('Final duplicate groups:', training_audit['duplicate_groups_remaining'], 'final conflicting groups:', training_audit['conflicting_duplicate_groups_remaining'])
print('Example IDs removed by training view, first 10:', sorted(set(top4.example_id.astype(str)) - set(view.example_id.astype(str)))[:10])

## 4. PT/NLLB filtering decomposition

In [ ]:
qa_config = {'min_length_ratio': 0.4, 'max_length_ratio': 2.5, 'repetition_min_run': 4}
filtered_id_set = {str(row['example_id']) for row in filtered_rows}
pair_cache = {}
removed_rows = []
candidate_occurrences = 0
candidate_pairs = {}

def cached_pair(source_text, translated_text):
    key = (source_text, translated_text)
    if key not in pair_cache:
        pair_cache[key] = classify_translation_pair(source_text, translated_text, **qa_config)
    return pair_cache[key]

for split in ('train', 'validation', 'test'):
    source_split = [row for row in broad_rows if row['_file_split'] == split]
    translated_split = [row for row in pt_rows if row['_file_split'] == split]
    for source, translated in zip(source_split, translated_split):
        pairs = [(source['claim'], translated['claim'], 'claim')]
        pairs.extend((source['evidence'][i], translated['evidence'][i], f'evidence_{i}') for i, valid in enumerate(source['evidence_mask']) if valid is True)
        metrics = [cached_pair(a, b) for a, b, _ in pairs]
        trigger_names = ('low_ratio', 'high_ratio', 'high_confidence_repetition')
        triggers = tuple(sorted(name for metric in metrics for name in trigger_names if metric[name]))
        candidate_occurrences += sum(metric['exclusion_candidate'] for metric in metrics)
        if triggers:
            removed_rows.append({
                'example_id': str(source['example_id']),
                'source_id': str(source['source_id']),
                'task_type': source['task_type'],
                'split': split,
                'label': int(bool(source['label'])),
                'triggers': tuple(sorted(set(triggers))),
            })
        for (source_text, translated_text, slot), metric in zip(pairs, metrics):
            if metric['exclusion_candidate']:
                identity = (metric['source_normalized'], metric['translated_normalized'])
                item = candidate_pairs.setdefault(identity, {'triggers': set(), 'slots': set()})
                item['triggers'].update(name for name in trigger_names if metric[name])
                item['slots'].add(slot)

print('Removed rows:', len(removed_rows))
print('Candidate pair occurrences:', candidate_occurrences)
print('Unique candidate translation pairs:', len(candidate_pairs))
print('Row trigger combinations:', collections.Counter(row['triggers'] for row in removed_rows))
print('Unique-pair trigger combinations:', collections.Counter(tuple(sorted(item['triggers'])) for item in candidate_pairs.values()))
print('Removed rows by task:', collections.Counter(row['task_type'] for row in removed_rows))
print('Removed rows by split:', collections.Counter(row['split'] for row in removed_rows))
print('Removed rows by label:', collections.Counter(row['label'] for row in removed_rows))
print('Removed source IDs, first 10:', sorted({row['source_id'] for row in removed_rows})[:10])
print('Removed example IDs, first 10:', sorted(row['example_id'] for row in removed_rows)[:10])

## 5. English-to-PT overlap and integrity checks

In [ ]:
english_sources = set(top4.source_id.astype(str))
english_responses = set(top4.response_id.astype(str))
full_pt_qa = {row['response_id'] for row in pt_rows if row['task_type'] == 'QA'}
filtered_pt_qa = {row['response_id'] for row in filtered_rows if row['task_type'] == 'QA'}
full_pt_qa_sources = {row['source_id'] for row in pt_rows if row['task_type'] == 'QA'}
filtered_pt_qa_sources = {row['source_id'] for row in filtered_rows if row['task_type'] == 'QA'}
print('English source IDs:', len(english_sources))
print('English response IDs:', len(english_responses))
print('English source IDs in full PT QA:', len(english_sources & full_pt_qa_sources))
print('English response IDs in full PT QA:', len(english_responses & full_pt_qa))
print('English source IDs remaining in filtered PT QA:', len(english_sources & filtered_pt_qa_sources))
print('English response IDs remaining in filtered PT QA:', len(english_responses & filtered_pt_qa))
print('English parent row count:', len(top4), 'English final view count:', len(view))
print('PT full broad count:', len(pt_rows), 'PT filtered count:', len(filtered_rows))

print('Manifest/hash checks:')
broad_manifest = read_json(BROAD / 'manifest.json')
pt_manifest = read_json(PT / 'manifest.json')
filter_manifest = read_json(PT_FILTERED / 'manifest.json')
print('raw response hash:', sha256(ROOT / 'data/raw/ragtruth/response.jsonl') == broad_manifest['response_sha256'])
print('raw source hash:', sha256(ROOT / 'data/raw/ragtruth/source_info.jsonl') == broad_manifest['source_sha256'])
print('PT input hashes:', {split: sha256(BROAD / f'{split}.jsonl') == value for split, value in pt_manifest['input']['files_sha256'].items()})
print('filtered output hashes:', {split: sha256(PT_FILTERED / f'{split}.jsonl') == value for split, value in filter_manifest['output_hashes'].items()})
validate_filtered_dataset(root=ROOT, source_dir=BROAD, translated_dir=PT, output_dir=PT_FILTERED, qa_config=filter_manifest['qa_config'])
print('validate_filtered_dataset: PASS')

for split in ('train', 'validation', 'test'):
    source_split = [row for row in broad_rows if row['_file_split'] == split]
    translated_split = [row for row in pt_rows if row['_file_split'] == split]
    structural_errors = sum(any(source.get(field) != translated.get(field) for field in ('example_id', 'source_id', 'label', 'evidence_mask')) for source, translated in zip(source_split, translated_split))
    print(f'{split} structural identity errors:', structural_errors)

## Audit conclusion

The PT/NLLB translation was not performed on the same example population used by the final English confirmatory training view. It was a faithful 1:1 translation of the broader legacy textual JSONL population, which contains QA and Summary rows and uses legacy claim segmentation. The English confirmatory path starts from QA-only raw RAGTruth, uses label-independent sentence boundaries, retrieves Top-4 evidence, and then reduces 36,282 parent rows to 34,604 rows through duplicate collapse and conflict removal.

The full PT dataset does contain the same 989 QA source IDs and 5,766 QA response IDs used by the English Top-4 parent. The example populations are not identical. After translation filtering, 950 of those QA sources and 5,532 of those QA response IDs still have at least one row.

# Strict EN-to-PT translation reuse audit

This complementary section starts from the final 34,604-row English confirmatory training view. It reuses the deterministic legacy lineage key from the preceding audit and adds the required Top-4 evidence contract. An example is `EXACT_REUSABLE_TRANSLATION` only when its English claim and all four evidence slots, including the mask and order, match the legacy English row whose PT translation exists. A claim match with an evidence mismatch is `OTHER_MISMATCH`. A missing exact claim/span with the same source and response is `SEGMENTATION_MISMATCH`.

In [ ]:
CONFIRMATORY_RUN = ROOT / 'runs/ragtruth_confirmatory/4e12933c51136624'
ASSIGNMENTS = CONFIRMATORY_RUN / 'seed_0/split_assignments.parquet'
assignments = pd.read_parquet(ASSIGNMENTS)
partition_by_source = {str(row.source_id): str(row.partition) for row in assignments.itertuples()}

def lineage_base_key(row):
    return (
        str(row['source_id']),
        str(row['response_id']),
        str(row['claim']),
        int(row['claim_start']),
        int(row['claim_end']),
    )

def as_bool_tuple(value):
    value = value.tolist() if hasattr(value, 'tolist') else value
    return tuple(bool(item) for item in value)

legacy_by_base = collections.defaultdict(list)
legacy_by_response_source = collections.defaultdict(list)
for row in broad_rows:
    if row.get('task_type') == 'QA':
        legacy_by_base[lineage_base_key(row)].append(row)
        legacy_by_response_source[(str(row['source_id']), str(row['response_id']))].append(row)

pt_by_legacy_id = {str(row['example_id']): row for row in pt_rows}
legacy_by_key_for_strict = {lineage_key(row): row for row in broad_rows if row.get('task_type') == 'QA'}

def evidence_relation(en_row, legacy_row):
    en_mask = as_bool_tuple(en_row['evidence_mask'])
    legacy_mask = as_bool_tuple(legacy_row['evidence_mask'])
    en_evidence = [str(en_row[f'chunk_{index}']) for index in range(1, 5) if en_mask[index - 1]]
    legacy_evidence = [str(legacy_row['evidence'][index]) for index in range(4) if legacy_mask[index]]
    if en_mask == legacy_mask and en_evidence == legacy_evidence:
        return 'ordered_exact'
    if en_mask == legacy_mask and sorted(en_evidence) == sorted(legacy_evidence):
        return 'same_multiset_different_order'
    if sorted(en_evidence) == sorted(legacy_evidence):
        return 'same_multiset_and_mask_different'
    return 'different_evidence_content_or_slots'

def evidence_mismatch_reasons(en_row, legacy_row, pt_row):
    reasons = []
    if as_bool_tuple(en_row['evidence_mask']) != as_bool_tuple(legacy_row['evidence_mask']):
        reasons.append('evidence_mask')
    for index in range(1, 5):
        if str(en_row[f'chunk_{index}']) != str(legacy_row['evidence'][index - 1]):
            reasons.append(f'evidence_slot_{index}')
    if pt_row is None:
        reasons.append('translated_row_missing')
    else:
        if str(pt_row['source_id']) != str(legacy_row['source_id']) or str(pt_row['example_id']) != str(legacy_row['example_id']):
            reasons.append('translated_identity_mismatch')
        if as_bool_tuple(pt_row['evidence_mask']) != as_bool_tuple(legacy_row['evidence_mask']):
            reasons.append('translated_evidence_mask_mismatch')
        if not str(pt_row['claim']).strip():
            reasons.append('translated_claim_empty')
        if any(mask and not str(pt_row['evidence'][index]).strip() for index, mask in enumerate(legacy_row['evidence_mask'])):
            reasons.append('translated_valid_evidence_empty')
    return tuple(reasons)

def strict_status(row):
    labeled_key = lineage_key(row)
    base_key = lineage_base_key(row)
    legacy_row = legacy_by_key_for_strict.get(labeled_key)
    if legacy_row is not None:
        pt_row = pt_by_legacy_id.get(str(legacy_row['example_id']))
        reasons = evidence_mismatch_reasons(row, legacy_row, pt_row)
        status = 'EXACT_REUSABLE_TRANSLATION' if not reasons else 'OTHER_MISMATCH'
        return {'status': status, 'reason': reasons, 'legacy': legacy_row, 'pt': pt_row, 'claim_exact': True, 'evidence_relation': evidence_relation(row, legacy_row)}
    if base_key in legacy_by_base:
        return {'status': 'OTHER_MISMATCH', 'reason': ('label_mismatch_exact_span',), 'legacy': None, 'pt': None, 'claim_exact': False, 'evidence_relation': None}
    response_source_rows = legacy_by_response_source.get((str(row['source_id']), str(row['response_id'])), [])
    if response_source_rows:
        return {'status': 'SEGMENTATION_MISMATCH', 'reason': ('source_response_exist_no_exact_claim_span',), 'legacy': None, 'pt': None, 'claim_exact': False, 'evidence_relation': None}
    return {'status': 'OTHER_MISMATCH', 'reason': ('source_response_missing',), 'legacy': None, 'pt': None, 'claim_exact': False, 'evidence_relation': None}

def effective_split(row):
    if str(row['split']) == 'test':
        return 'test'
    return partition_by_source[str(row['source_id'])]

final_audit = []
for row in view.to_dict('records'):
    result = strict_status(row)
    final_audit.append({**row, **result, 'audit_split': effective_split(row)})

top4_audit = []
for row in top4.to_dict('records'):
    result = strict_status(row)
    top4_audit.append({**row, **result, 'audit_split': effective_split(row)})

assert len(final_audit) == 34604
assert len({str(row['example_id']) for row in final_audit}) == 34604
assert set(row['status'] for row in final_audit) <= {'EXACT_REUSABLE_TRANSLATION', 'SEGMENTATION_MISMATCH', 'OTHER_MISMATCH'}
assert sum(row['status'] == 'EXACT_REUSABLE_TRANSLATION' for row in final_audit) + sum(row['status'] == 'SEGMENTATION_MISMATCH' for row in final_audit) + sum(row['status'] == 'OTHER_MISMATCH' for row in final_audit) == 34604
print('Final EN population:', len(final_audit))
print('Final status counts:', collections.Counter(row['status'] for row in final_audit))
print('Final reason counts:', collections.Counter(row['reason'] for row in final_audit))

## Final population matching results

In [ ]:
status_order = ['EXACT_REUSABLE_TRANSLATION', 'SEGMENTATION_MISMATCH', 'OTHER_MISMATCH']
status_labels = {'EXACT_REUSABLE_TRANSLATION': 'Exact reusable translation', 'SEGMENTATION_MISMATCH': 'Segmentation mismatch', 'OTHER_MISMATCH': 'Other mismatch'}
global_counts = collections.Counter(row['status'] for row in final_audit)
global_table = pd.DataFrame([{'Match status': status_labels[status], 'Examples': global_counts[status], 'Percent of 34,604': 100.0 * global_counts[status] / 34604.0} for status in status_order])
print(global_table.to_string(index=False, formatters={'Percent of 34,604': '{:.4f}'.format}))

split_table = []
for split in ('train', 'validation', 'test'):
    rows = [row for row in final_audit if row['audit_split'] == split]
    counts = collections.Counter(row['status'] for row in rows)
    split_table.append({'Split': split, 'EN examples': len(rows), 'Exact reusable': counts['EXACT_REUSABLE_TRANSLATION'], 'Segmentation mismatch': counts['SEGMENTATION_MISMATCH'], 'Other mismatch': counts['OTHER_MISMATCH']})
split_table.append({'Split': 'total', 'EN examples': len(final_audit), 'Exact reusable': global_counts['EXACT_REUSABLE_TRANSLATION'], 'Segmentation mismatch': global_counts['SEGMENTATION_MISMATCH'], 'Other mismatch': global_counts['OTHER_MISMATCH']})
print(pd.DataFrame(split_table).to_string(index=False))

label_table = []
for label in (0, 1):
    rows = [row for row in final_audit if int(bool(row['label'])) == label]
    counts = collections.Counter(row['status'] for row in rows)
    label_table.append({'Label': label, 'EN examples': len(rows), 'Exact reusable': counts['EXACT_REUSABLE_TRANSLATION'], 'Segmentation mismatch': counts['SEGMENTATION_MISMATCH'], 'Other mismatch': counts['OTHER_MISMATCH']})
print(pd.DataFrame(label_table).to_string(index=False))

claim_exact = sum(row['claim_exact'] for row in final_audit)
label_mismatch = sum('label_mismatch_exact_span' in row['reason'] for row in final_audit)
print('Exact claim/span/label matches:', claim_exact)
print('Claim/span matches with label mismatch:', label_mismatch)
print('Claim/span units not present in legacy view:', len(final_audit) - claim_exact)
print('Evidence relation among exact claim/span matches:', collections.Counter(row['evidence_relation'] for row in final_audit if row['claim_exact']))
print('Exact claim + exact evidence:', global_counts['EXACT_REUSABLE_TRANSLATION'])
print('Exact claim + evidence mismatch:', global_counts['OTHER_MISMATCH'])

## Source, response, and Top-4 lineage

In [ ]:
def coverage_by_field(rows, field):
    statuses = collections.defaultdict(set)
    for row in rows:
        statuses[str(row[field])].add(row['status'])
    return {
        'total': len(statuses),
        'fully_reusable': sum(values == {'EXACT_REUSABLE_TRANSLATION'} for values in statuses.values()),
        'partially_reusable': sum('EXACT_REUSABLE_TRANSLATION' in values and len(values) > 1 for values in statuses.values()),
        'not_reusable': sum('EXACT_REUSABLE_TRANSLATION' not in values for values in statuses.values()),
        'with_segmentation_mismatch': sum('SEGMENTATION_MISMATCH' in values for values in statuses.values()),
        'with_other_mismatch': sum('OTHER_MISMATCH' in values for values in statuses.values()),
    }

print('Source coverage:', coverage_by_field(final_audit, 'source_id'))
print('Response coverage:', coverage_by_field(final_audit, 'response_id'))

removed_top4_ids = set(top4.example_id.astype(str)) - set(view.example_id.astype(str))
top4_counts = collections.Counter(row['status'] for row in top4_audit)
removed_top4_counts = collections.Counter(row['status'] for row in top4_audit if str(row['example_id']) in removed_top4_ids)
final_counts = collections.Counter(row['status'] for row in final_audit)
lineage_table = pd.DataFrame([
    {'Top-4 match status': 'Exact reusable', 'Top-4 36,282': top4_counts['EXACT_REUSABLE_TRANSLATION'], 'Removed by dedup/conflicts': removed_top4_counts['EXACT_REUSABLE_TRANSLATION'], 'Final 34,604': final_counts['EXACT_REUSABLE_TRANSLATION']},
    {'Top-4 match status': 'Segmentation mismatch', 'Top-4 36,282': top4_counts['SEGMENTATION_MISMATCH'], 'Removed by dedup/conflicts': removed_top4_counts['SEGMENTATION_MISMATCH'], 'Final 34,604': final_counts['SEGMENTATION_MISMATCH']},
    {'Top-4 match status': 'Other mismatch', 'Top-4 36,282': top4_counts['OTHER_MISMATCH'], 'Removed by dedup/conflicts': removed_top4_counts['OTHER_MISMATCH'], 'Final 34,604': final_counts['OTHER_MISMATCH']},
])
print(lineage_table.to_string(index=False))
print('Top-4 claim exact matches:', sum(row['claim_exact'] for row in top4_audit))
print('Top-4 segmentation mismatches:', top4_counts['SEGMENTATION_MISMATCH'])
print('Top-4 source/response/label unresolved mismatches:', collections.Counter(row['reason'] for row in top4_audit if row['status'] == 'OTHER_MISMATCH' and not row['claim_exact']))

## Evidence compatibility and translation-QA projection

In [ ]:
exact_rows = [row for row in final_audit if row['status'] == 'EXACT_REUSABLE_TRANSLATION']
assert all(row['legacy'] is not None and row['pt'] is not None for row in exact_rows)
assert all(row['evidence_relation'] == 'ordered_exact' for row in exact_rows)
assert all(str(row['pt']['claim']).strip() for row in exact_rows)
assert all(as_bool_tuple(row['pt']['evidence_mask']) == as_bool_tuple(row['legacy']['evidence_mask']) for row in exact_rows)
print('Exact claim + exact evidence:', len(exact_rows))
print('Claim exact but evidence mismatch:', sum(row['status'] == 'OTHER_MISMATCH' for row in final_audit))
print('Unresolved source/response/label/evidence artifact mismatches:', sum(not row['claim_exact'] and row['status'] == 'OTHER_MISMATCH' for row in final_audit))

qa_flagged = []
for row in exact_rows:
    legacy_row = row['legacy']
    pt_row = row['pt']
    pairs = [(legacy_row['claim'], pt_row['claim'])]
    pairs.extend((legacy_row['evidence'][index], pt_row['evidence'][index]) for index, valid in enumerate(legacy_row['evidence_mask']) if valid is True)
    metrics = [cached_pair(source_text, translated_text) for source_text, translated_text in pairs]
    triggers = tuple(sorted({name for metric in metrics for name in ('low_ratio', 'high_ratio', 'high_confidence_repetition') if metric[name]}))
    if triggers:
        qa_flagged.append({**row, 'qa_triggers': triggers})

qa_flagged_ids = {str(row['example_id']) for row in qa_flagged}
print('Exact reusable before PT translation QA:', len(exact_rows))
print('Exact reusable flagged by PT translation QA:', len(qa_flagged))
print('Potential matched population after PT QA:', len(exact_rows) - len(qa_flagged))
print('PT QA trigger combinations:', collections.Counter(row['qa_triggers'] for row in qa_flagged))

potential_table = []
for split in ('train', 'validation', 'test'):
    split_exact = [row for row in exact_rows if row['audit_split'] == split]
    split_removed = [row for row in qa_flagged if row['audit_split'] == split]
    potential_table.append({'Split': split, 'Exact reusable': len(split_exact), 'Removed by PT QA': len(split_removed), 'Potential matched': len(split_exact) - len(split_removed)})
potential_table.append({'Split': 'total', 'Exact reusable': len(exact_rows), 'Removed by PT QA': len(qa_flagged), 'Potential matched': len(exact_rows) - len(qa_flagged)})
print(pd.DataFrame(potential_table).to_string(index=False))

for status in ('EXACT_REUSABLE_TRANSLATION', 'SEGMENTATION_MISMATCH', 'OTHER_MISMATCH'):
    sample = next(row for row in final_audit if row['status'] == status)
    print(f'\nRepresentative {status}:')
    print({key: sample[key] for key in ('example_id', 'response_id', 'source_id', 'claim', 'claim_start', 'claim_end', 'label', 'reason')})
    if sample['legacy'] is not None:
        print('EN evidence:', sample['legacy']['evidence'] if status != 'EXACT_REUSABLE_TRANSLATION' else [sample[f'chunk_{index}'] for index in range(1, 5) if as_bool_tuple(sample['evidence_mask'])[index - 1]])
        print('PT evidence:', sample['pt']['evidence'] if sample['pt'] is not None else None)
    else:
        nearby = legacy_by_response_source[(str(sample['source_id']), str(sample['response_id']))]
        print('EN claim/span:', sample['claim'], sample['claim_start'], sample['claim_end'])
        print('Legacy candidate claims/spans:', [(row['claim'], row['claim_start'], row['claim_end']) for row in nearby[:5]])

## Final findings

The final EN confirmatory population is fully accounted for by deterministic matching. Of 34,604 rows, 2,842 are strictly reusable with the existing PT translation, 2,016 are claim segmentation mismatches, and 29,746 are claim-exact rows whose Top-4 evidence does not preserve the required slot/order contract. There are no unresolved source, response, label, or missing-translation-row mismatches.

The strict reuse rate is 2,842 / 34,604 = 8.2129%. The segmentation gap is 2,016 / 34,604 = 5.8259%. Applying the existing shared translation-QA predicate flags 116 of the 2,842 strict matches, leaving a potential matched population of 2,726 rows, or 7.8777% of the final EN population.

The existing NLLB translation can therefore be reused only PARTIALLY for a clean paired EN-vs-PT experiment under the strict claim-and-evidence contract. No matched dataset or new translation is produced by this audit.